# NeuroTwin feature-set selection
Exploratory grouped validation on Brighter Time (Didikoglu and colleagues, DOI 10.48420/20212265, CC BY 4.0). No new untouched final test. Does not replace the deployed model.


In [ ]:
from google.colab import files
uploaded=files.upload()
if len(uploaded)!=1: raise ValueError("Upload only BrighterTimeData.csv")
csv_path=next(iter(uploaded))


In [ ]:
"""Exploratory grouped validation. Does not overwrite the deployed model."""
import json
from pathlib import Path
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error


import pandas as pd
FEATURES=['previous_reaction_ms', 'recent_reaction_mean_ms', 'previous_time_awake', 'previous_sleepiness', 'previous_clock_sin', 'previous_clock_cos']
def prepare(csv_path):
    raw = pd.read_csv(csv_path)
    required = ['id','task','date','time_of_day','time_awake','kss_score','medianrt']
    if not set(required).issubset(raw.columns):
        raise ValueError('This is not the expected Brighter Time CSV.')
    data = raw.loc[raw.task.eq('PVT'), required].copy()
    for column in ['time_of_day','time_awake','kss_score','medianrt']:
        data[column] = pd.to_numeric(data[column], errors='coerce')
    data['timestamp'] = pd.to_datetime(data['date'], errors='coerce') + pd.to_timedelta(data.time_of_day, unit='h')
    valid = data[required].notna().all(axis=1) & data.timestamp.notna() & data.time_of_day.between(0,24,inclusive='left') & data.time_awake.ge(0) & data.medianrt.gt(0)
    rejected = int((~valid).sum())
    data = data.loc[valid].sort_values(['id','timestamp']).copy()
    if data.duplicated(['id','timestamp']).any():
        raise ValueError('Duplicate participant timestamps require review.')
    group = data.groupby('id', sort=False)
    data['previous_reaction_ms'] = group.medianrt.shift(1)
    data['recent_reaction_mean_ms'] = group.medianrt.transform(lambda s:s.shift(1).rolling(3,min_periods=1).mean())
    data['previous_time_awake'] = group.time_awake.shift(1)
    data['previous_sleepiness'] = group.kss_score.shift(1)
    previous_clock = group.time_of_day.shift(1)
    data['previous_clock_sin'] = np.sin(2*np.pi*previous_clock/24)
    data['previous_clock_cos'] = np.cos(2*np.pi*previous_clock/24)
    previous_timestamp = group.timestamp.shift(1)
    data['gap_hours'] = (data.timestamp-previous_timestamp).dt.total_seconds()/3600
    # The next recorded session need not be tomorrow. Exclude gaps over 48h
    # as an explicit experiment design choice, not a clinical threshold.
    examples = data.loc[data.gap_hours.gt(0) & data.gap_hours.le(48)].dropna(subset=FEATURES).copy()
    audit = {'raw_rows':len(raw),'pvt_rows':int(raw.task.eq('PVT').sum()),'invalid_pvt_rows':rejected,'eligible_examples':len(examples),'participants':int(examples.id.nunique()),'observed_sleepiness_range':[float(data.kss_score.min()),float(data.kss_score.max())]}
    return examples, audit

SETS={
 'History only':FEATURES[:2],
 'History + context':FEATURES[:4],
 'History + context + clock':FEATURES,
}

def compare_features(csv_path,output_path):
 data,audit=prepare(csv_path)
 folds=list(GroupKFold(n_splits=5).split(data,groups=data.id))
 candidates=[]
 for label,features in SETS.items():
  for alpha in [.1,1.,10.,100.]:
   scores=[]
   for train_index,validation_index in folds:
    training,validation=data.iloc[train_index],data.iloc[validation_index]
    assert set(training.id).isdisjoint(validation.id)
    fitted=make_pipeline(StandardScaler(),Ridge(alpha=alpha))
    fitted.fit(training[features],training.medianrt)
    scores.append(float(mean_absolute_error(validation.medianrt,fitted.predict(validation[features]))))
   candidates.append({'feature_set':label,'features':features,'alpha':alpha,'fold_mae_ms':scores,'mean_validation_mae_ms':float(np.mean(scores))})
 best_per_set=[min((row for row in candidates if row['feature_set']==label),key=lambda row:row['mean_validation_mae_ms']) for label in SETS]
 selected=min(best_per_set,key=lambda row:row['mean_validation_mae_ms'])
 baseline=[]
 for _,index in folds:
  validation=data.iloc[index]
  baseline.append(float(mean_absolute_error(validation.medianrt,validation.recent_reaction_mean_ms)))
 result={
  'stage':'exploratory-feature-selection','source_doi':'10.48420/20212265','audit':audit,
  'method':'Five participant-disjoint GroupKFold validation folds; scaler fitted inside each fold; feature set and alpha selected by mean validation MAE.',
  'selected':selected,'best_per_set':best_per_set,'all_candidates':candidates,
  'recent_average_validation_mae_ms':float(np.mean(baseline)),
  'independent_final_test':False,'changes_deployed_model':False,
  'limitations':['Dataset already explored in prior evaluations. These are selection/validation scores, not an untouched final test.','Fold scores equally weighted; participants are disjoint within each split.','Prior sessions supply inputs for held-out participants, never fit targets.','Source modified sleepiness scale is 1–10.','No clinical validation or validation for the NeuroTwin five-trial task.'],
 }
 Path(output_path).write_text(json.dumps(result,indent=2,allow_nan=False),encoding='utf-8')
 print(json.dumps({'selected_feature_set':selected['feature_set'],'selected_alpha':selected['alpha'],'validation_mae_ms':selected['mean_validation_mae_ms'],'recent_average_validation_mae_ms':result['recent_average_validation_mae_ms'],'independent_final_test':False,'deployed_model_changed':False},indent=2))
 return result



In [ ]:
result=compare_features(csv_path,"feature-selection.json")
files.download("feature-selection.json")
